In [6]:
import pandas as pd
import pathlib
from datetime import datetime
import os
import csv

# ------------------------------------------------------------
# 1. Загрузка всех CSV из папки "Входящие" в один DataFrame
# ------------------------------------------------------------
input_dir = pathlib.Path("Входящие")
all_dfs = []
encodings = ['cp1251', 'utf-8', 'latin1', 'cp866']

for file_path in input_dir.glob("*"):
    if file_path.suffix.lower() != '.csv':
        continue

    # Пробуем разные кодировки и разделители
    for enc in encodings:
        try:
            # Сначала определим разделитель автоматически
            with open(file_path, 'r', encoding=enc) as f:
                sample = f.read(4096)
                sniffer = csv.Sniffer()
                delimiter = sniffer.sniff(sample).delimiter
            # Читаем с найденным разделителем, пропускаем плохие строки
            df = pd.read_csv(file_path, encoding=enc, sep=delimiter,
                             on_bad_lines='skip', engine='python')
            all_dfs.append(df)
            print(f"Загружен {file_path.name} (разделитель '{delimiter}', кодировка {enc})")
            break
        except (UnicodeDecodeError, csv.Error, pd.errors.ParserError) as e:
            print(f"Не удалось прочитать {file_path.name} c {enc}: {e}")
            continue
    else:
        print(f"Файл {file_path.name} не загружен ни в одной кодировке/разделителе")

if not all_dfs:
    raise ValueError("Нет загруженных CSV файлов в папке Входящие")

sbis_df = pd.concat(all_dfs, ignore_index=True)

# Новые имена столбцов по заданию (30 штук)
new_columns = [
    "Дата", "Номер", "Сумма", "Статус", "Примечание", "Комментарий",
    "Контрагент", "ИНН/КПП", "Организация", "ИНН/КПП", "Тип документа",
    "Имя файла", "Дата", "Номер 1", "Сумма 1", "Сумма НДС",
    "Ответственный", "Подразделение", "Код", "Дата", "Время",
    "Тип пакета", "Идентификатор пакета", "Запущено в обработку",
    "Получено контрагентом", "Завершено", "Увеличение суммы",
    "НДC", "Уменьшение суммы", "НДС"
]

current_cols = len(sbis_df.columns)
if current_cols == len(new_columns):
    sbis_df.columns = new_columns
elif current_cols > len(new_columns):
    # Добавляем временные имена для лишних столбцов
    extra = [f"Доп_{i}" for i in range(1, current_cols - len(new_columns) + 1)]
    sbis_df.columns = new_columns + extra
else:
    # Если столбцов меньше, обрезаем список
    sbis_df.columns = new_columns[:current_cols]

# ------------------------------------------------------------
# 2. Обработка файлов аптек
# ------------------------------------------------------------
pharmacy_dir = pathlib.Path("Аптеки/csv/correct")
output_base = pathlib.Path("Результат")
today_str = datetime.now().strftime("%Y-%m-%d")
output_date_dir = output_base / today_str
output_date_dir.mkdir(parents=True, exist_ok=True)

output_columns_order = [
    '№ п/п', 'Штрих-код партии', 'Наименование товара', 'Поставщик',
    'Дата приходного документа', 'Номер приходного документа',
    'Дата накладной', 'Номер накладной', 'Номер счет-фактуры',
    'Сумма счет-фактуры', 'Кол-во',
    'Сумма в закупочных ценах без НДС', 'Ставка НДС поставщика',
    'Сумма НДС', 'Сумма в закупочных ценах с НДС', 'Дата счет-фактуры',
    'Сравнение дат'
]

for file_path in pharmacy_dir.glob("*.csv"):
    # Определяем кодировку для файла аптеки
    for enc in encodings:
        try:
            apt_df = pd.read_csv(file_path, encoding=enc)
            break
        except UnicodeDecodeError:
            continue
    else:
        print(f"Не удалось прочитать {file_path.name}")
        continue

    apt_df['Номер счет-фактуры'] = ''
    apt_df['Сумма счет-фактуры'] = ''
    apt_df['Дата счет-фактуры'] = ''
    apt_df['Сравнение дат'] = ''

    for idx, row in apt_df.iterrows():
        # Если поставщик "ЕАПТЕКА", модифицируем номер накладной
        if row.get('Поставщик') == 'ЕАПТЕКА':
            original_number = str(row['Номер накладной'])
            modified_number = f"{original_number}/15"
        else:
            modified_number = str(row['Номер накладной'])

        # Поиск в sbis_df
        mask = sbis_df['Номер'] == modified_number
        matches = sbis_df[mask]
        allowed_types = ["СчФктр", "УпдДоп", "УпдСчфДоп", "ЭДОНакл"]
        filtered = matches[matches['Тип документа'].isin(allowed_types)]

        if not filtered.empty:
            first_match = filtered.iloc[0]
            inv_number = first_match['Номер']
            inv_sum = first_match['Сумма']
            inv_date_raw = first_match['Дата']

            # Преобразуем дату
            try:
                for fmt in ('%Y-%m-%d', '%d.%m.%Y', '%Y/%m/%d', '%d/%m/%Y'):
                    try:
                        inv_date_dt = datetime.strptime(str(inv_date_raw), fmt)
                        break
                    except:
                        continue
                else:
                    inv_date_formatted = str(inv_date_raw)
                inv_date_formatted = inv_date_dt.strftime('%d.%m.%Y')
            except:
                inv_date_formatted = str(inv_date_raw)

            apt_df.at[idx, 'Номер счет-фактуры'] = inv_number
            apt_df.at[idx, 'Сумма счет-фактуры'] = inv_sum
            apt_df.at[idx, 'Дата счет-фактуры'] = inv_date_formatted

            # Сравнение дат
            doc_date_raw = row.get('Дата накладной', '')
            if pd.isna(doc_date_raw) or doc_date_raw == '':
                compare = ''
            else:
                try:
                    for fmt in ('%Y-%m-%d', '%d.%m.%Y', '%Y/%m/%d', '%d/%m/%Y'):
                        try:
                            doc_date_dt = datetime.strptime(str(doc_date_raw), fmt)
                            break
                        except:
                            continue
                    else:
                        doc_date_dt = None
                    if doc_date_dt is not None:
                        doc_date_formatted = doc_date_dt.strftime('%d.%m.%Y')
                    else:
                        doc_date_formatted = str(doc_date_raw)
                except:
                    doc_date_formatted = str(doc_date_raw)

                if inv_date_formatted != doc_date_formatted:
                    compare = "Не совпадает!"
                else:
                    compare = ""
            apt_df.at[idx, 'Сравнение дат'] = compare

    # Формируем итоговый DataFrame
    for col in output_columns_order:
        if col not in apt_df.columns:
            apt_df[col] = ''

    result_df = apt_df[output_columns_order].copy()
    output_filename = f"{file_path.stem} - результат.xlsx"
    output_path = output_date_dir / output_filename
    result_df.to_excel(output_path, index=False)
    print(f"Обработан {file_path.name} -> {output_path}")

print("Готово!")

Загружен Входящие 03.csv (разделитель ';', кодировка cp1251)
Загружен Входящие 01.csv (разделитель '/', кодировка cp1251)
Загружен Входящие 02.csv (разделитель ';', кодировка cp1251)
Готово!
